# Local Secure Dedup Demo Walkthrough

This notebook is designed for complete reviewer transparency.

It keeps everything visible in one place:

- the current backend status of the live API,
- the collected test cases,
- the full test source files,
- the saved test reports and raw outputs,
- optional rerun cells for the main suites,
- the benchmark script source and comparison artifacts,
- the live PoW / chunk-sharing / rate-limit demo,
- the dataset and model metrics behind the behavioural layer.

Backend note:

- The repo supports `Redis` for index / policy / reputation state, with in-memory fallback.
- The repo supports `LocalStack` / S3-style chunk storage, with filesystem fallback.
- The first cells below show what the currently running server is actually using.

In [29]:
import json
import subprocess
import sys
import tempfile
import time
from html import escape
from pathlib import Path

import pandas as pd
import requests
from IPython.display import HTML, Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'app.py').exists() and (candidate / 'docs').exists():
            return candidate
    raise RuntimeError('Could not find repo root from the notebook working directory.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())

def choose_repo_python() -> str:
    candidates = [
        REPO_ROOT / '.venv' / 'Scripts' / 'python.exe',
        REPO_ROOT / '.venv' / 'bin' / 'python',
        Path(sys.executable),
    ]
    for candidate in candidates:
        if Path(candidate).exists():
            return str(candidate)
    return sys.executable

REPO_PYTHON = choose_repo_python()
BASE_URL = 'http://127.0.0.1:8000'
API_KEY = 'dev-api-key'
CLIENT_ID = f'notebook-demo-{int(time.time())}'
HEADERS = {'X-API-Key': API_KEY, 'X-Client-ID': CLIENT_ID}

def run_and_show(args, cwd=REPO_ROOT):
    print('COMMAND:', ' '.join(str(arg) for arg in args))
    result = subprocess.run(args, cwd=str(cwd), capture_output=True, text=True, check=False)
    print('RETURN CODE:', result.returncode)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('STDERR:')
        print(result.stderr)
    return result

def show_text_file(path: Path, language: str = 'text'):
    text = path.read_text(encoding='utf-8')
    numbered = '\n'.join(f'{idx:4}: {line}' for idx, line in enumerate(text.splitlines(), start=1))
    rel = path.relative_to(REPO_ROOT)
    display(HTML(f'<h4>{escape(str(rel))}</h4><pre>{escape(numbered)}</pre>'))

print('REPO_ROOT   =', REPO_ROOT)
print('REPO_PYTHON =', REPO_PYTHON)
print('BASE_URL    =', BASE_URL)
print('CLIENT_ID   =', CLIENT_ID)

REPO_ROOT   = E:\secure-dedup
REPO_PYTHON = E:\secure-dedup\.venv\Scripts\python.exe
BASE_URL    = http://127.0.0.1:8000
CLIENT_ID   = notebook-demo-1774323266


In [30]:
print('Docker compose status:')
run_and_show(['docker', 'compose', '-f', str(REPO_ROOT / 'docker-compose.local.yml'), 'ps'])

health = requests.get(f'{BASE_URL}/health', timeout=30)
health.raise_for_status()
config = requests.get(f'{BASE_URL}/demo/config', timeout=30)
config.raise_for_status()
config = config.json()

summary = {
    'health': health.json(),
    'demo_mode': config['demo_mode'],
    'storage_backend': config['storage']['backend'],
    'configured_storage_backend': config['storage']['configured_backend'],
    'storage_fallback_active': config['storage']['fallback_active'],
    'storage_endpoint': config['storage']['endpoint'],
    'storage_bucket': config['storage']['bucket'],
    'fingerprint_mode': config['fingerprint']['mode'],
    'detection_mode': config['detection']['mode'],
}
print(json.dumps(summary, indent=2))

print('\nLocalStack bucket visibility:')
run_and_show(['docker', 'exec', 'secure_dedup_localstack', 'awslocal', 's3', 'ls'])

print('Redis visibility:')
run_and_show(['docker', 'exec', 'secure_dedup_redis', 'redis-cli', 'ping'])

Docker compose status:
COMMAND: docker compose -f E:\secure-dedup\docker-compose.local.yml ps
RETURN CODE: 0
NAME                      IMAGE                          COMMAND                  SERVICE      CREATED       STATUS                  PORTS
secure_dedup_localstack   localstack/localstack:latest   "docker-entrypoint.sh"   localstack   5 weeks ago   Up 36 hours (healthy)   0.0.0.0:4566->4566/tcp, [::]:4566->4566/tcp
secure_dedup_redis        redis:7-alpine                 "docker-entrypoint.sâ€¦"   redis        5 weeks ago   Up 36 hours             0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp

{
  "health": {
    "status": "ok"
  },
  "demo_mode": true,
  "storage_backend": "localstack",
  "configured_storage_backend": "localstack",
  "storage_fallback_active": false,
  "storage_endpoint": "http://127.0.0.1:4566",
  "storage_bucket": "chunks",
  "fingerprint_mode": "secret_hmac",
  "detection_mode": "supervised"
}

LocalStack bucket visibility:
COMMAND: docker exec secure_dedup_loc

CompletedProcess(args=['docker', 'exec', 'secure_dedup_redis', 'redis-cli', 'ping'], returncode=0, stdout='PONG\n', stderr='')

## Saved Test Reports

These are the persisted human-readable reports from the latest saved pytest runs.

In [31]:
display(Markdown((REPO_ROOT / 'test_reports' / 'frequency_attack_pytest_20260322_181202.md').read_text(encoding='utf-8')))
display(Markdown((REPO_ROOT / 'test_reports' / 'attack_detection_demo_pytest_20260322_181200.md').read_text(encoding='utf-8')))

# Frequency Attack Pytest Report

- Generated: `2026-03-22 18:12:02`
- Suite: `tests/test_frequency_attack_resistance.py`
- Result: `15/15 passed`

## Saved Artifacts

- Raw pytest output: `test_reports/frequency_attack_pytest_20260322_181202.txt`
- JUnit XML: `test_reports/frequency_attack_pytest_20260322_181202.xml`

## What This Suite Demonstrates

- public `SHA-256` tokens are reproducible by an adversary,
- `HMAC-SHA256` tokens are not reproducible without the server secret,
- cross-user dedup is preserved,
- frequency analysis and confirmation attacks are blocked at the token level,
- HKDF-derived chunk keys remain bound to the secret-assisted token.


# Behavioural Attack Demo Pytest Report

- Generated: `2026-03-22 18:12:00`
- Suite: `tests/test_attack_detection_demo.py`
- Result: `11/11 passed`

## Saved Artifacts

- Raw pytest output: `test_reports/attack_detection_demo_pytest_20260322_181200.txt`
- JUnit XML: `test_reports/attack_detection_demo_pytest_20260322_181200.xml`

## What This Suite Demonstrates

- hash probing is detected and rate limited,
- dedup DoS is detected and blocked,
- ownership fraud through repeated PoW attempts is detected,
- legitimate low-rate or low-failure behaviour is not misclassified,
- the REFA gap demonstration shows:
  `REFA would: ALLOW | Our framework would: RATE_LIMIT`.


## Collected Test Cases

This cell lists the individual tests so reviewers can see the exact case names before looking at the source.

In [32]:
run_and_show([
    REPO_PYTHON,
    '-m',
    'pytest',
    'tests/test_frequency_attack_resistance.py',
    'tests/test_attack_detection_demo.py',
    'tests/test_encryption.py',
    '--collect-only',
    '-q',
])

COMMAND: E:\secure-dedup\.venv\Scripts\python.exe -m pytest tests/test_frequency_attack_resistance.py tests/test_attack_detection_demo.py tests/test_encryption.py --collect-only -q
RETURN CODE: 0
tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_sha256_token_is_reproducible_by_adversary
tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_sha256
tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_wrong_key
tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_reproducible_with_correct_key
tests/test_frequency_attack_resistance.py::TestDeduplicationPreserved::test_ten_users_same_chunk_same_hmac_token
tests/test_frequency_attack_resistance.py::TestDeduplicationPreserved::test_dedup_savings_identical_across_schemes
tests/test_frequency_attack_resistance.py::TestFrequencyAnalysisAttack::test_sha256_frequency_leaks_to

CompletedProcess(args=['E:\\secure-dedup\\.venv\\Scripts\\python.exe', '-m', 'pytest', 'tests/test_frequency_attack_resistance.py', 'tests/test_attack_detection_demo.py', 'tests/test_encryption.py', '--collect-only', '-q'], returncode=0, stdout='tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_sha256_token_is_reproducible_by_adversary\ntests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_sha256\ntests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_wrong_key\ntests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_reproducible_with_correct_key\ntests/test_frequency_attack_resistance.py::TestDeduplicationPreserved::test_ten_users_same_chunk_same_hmac_token\ntests/test_frequency_attack_resistance.py::TestDeduplicationPreserved::test_dedup_savings_identical_across_schemes\ntests/test_frequency_attack_resistance.py::Tes

## Optional Transparent Reruns

These cells rerun the main suites directly from the notebook.

In [ ]:
run_and_show([REPO_PYTHON, '-m', 'pytest', 'tests/test_frequency_attack_resistance.py', '-v', '-s'])

COMMAND: E:\secure-dedup\.venv\Scripts\python.exe -m pytest tests/test_frequency_attack_resistance.py -v -s
RETURN CODE: 0
============================= test session starts =============================
platform win32 -- Python 3.12.4, pytest-9.0.2, pluggy-1.6.0 -- E:\secure-dedup\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: E:\secure-dedup
plugins: anyio-4.12.1
collecting ... collected 15 items

tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_sha256_token_is_reproducible_by_adversary PASSED
tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_sha256 PASSED
tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_wrong_key PASSED
tests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_reproducible_with_correct_key PASSED
tests/test_frequency_attack_resistance.py::TestDeduplicationPreserved::test_ten_users

CompletedProcess(args=['E:\\secure-dedup\\.venv\\Scripts\\python.exe', '-m', 'pytest', 'tests/test_frequency_attack_resistance.py', '-v', '-s'], returncode=0, stdout='\x1b============================= test session starts =============================\x1b\nplatform win32 -- Python 3.12.4, pytest-9.0.2, pluggy-1.6.0 -- E:\\secure-dedup\\.venv\\Scripts\\python.exe\ncachedir: .pytest_cache\nrootdir: E:\\secure-dedup\nplugins: anyio-4.12.1\n\x1bcollecting ... \x1bcollected 15 items\n\ntests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_sha256_token_is_reproducible_by_adversary \x1bPASSED\x1b\ntests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_sha256 \x1bPASSED\x1b\ntests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_not_reproducible_with_wrong_key \x1bPASSED\x1b\ntests/test_frequency_attack_resistance.py::TestTokenNonReproducibility::test_hmac_token_reproducible_with_corre

In [ ]:
run_and_show([REPO_PYTHON, '-m', 'pytest', 'tests/test_attack_detection_demo.py', '-v', '-s'])

COMMAND: E:\secure-dedup\.venv\Scripts\python.exe -m pytest tests/test_attack_detection_demo.py -v -s
RETURN CODE: 0
============================= test session starts =============================
platform win32 -- Python 3.12.4, pytest-9.0.2, pluggy-1.6.0 -- E:\secure-dedup\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: E:\secure-dedup
plugins: anyio-4.12.1
collecting ... collected 11 items

tests/test_attack_detection_demo.py::TestHashProbingDetection::test_hash_probing_detected 
[hash_probing] upload_to_query_ratio = 0.020
[hash_probing] LABEL=hash_probing  ACTION=RATE_LIMIT
PASSED
tests/test_attack_detection_demo.py::TestHashProbingDetection::test_normal_client_not_flagged 
[normal] upload_to_query_ratio = 10.000
[normal] LABEL=normal  ACTION=ALLOW
PASSED
tests/test_attack_detection_demo.py::TestHashProbingDetection::test_hash_probing_adaptive_pow_increases_difficulty 
[adaptive_pow] attacker risk=0.30  challenge_length=70
[adaptive_pow] normal   risk=0.00  challenge_len

CompletedProcess(args=['E:\\secure-dedup\\.venv\\Scripts\\python.exe', '-m', 'pytest', 'tests/test_attack_detection_demo.py', '-v', '-s'], returncode=0, stdout='\x1b============================= test session starts =============================\x1b\nplatform win32 -- Python 3.12.4, pytest-9.0.2, pluggy-1.6.0 -- E:\\secure-dedup\\.venv\\Scripts\\python.exe\ncachedir: .pytest_cache\nrootdir: E:\\secure-dedup\nplugins: anyio-4.12.1\n\x1bcollecting ... \x1bcollected 11 items\n\ntests/test_attack_detection_demo.py::TestHashProbingDetection::test_hash_probing_detected \n[hash_probing] upload_to_query_ratio = 0.020\n[hash_probing] LABEL=hash_probing  ACTION=RATE_LIMIT\n\x1bPASSED\x1b\ntests/test_attack_detection_demo.py::TestHashProbingDetection::test_normal_client_not_flagged \n[normal] upload_to_query_ratio = 10.000\n[normal] LABEL=normal  ACTION=ALLOW\n\x1bPASSED\x1b\ntests/test_attack_detection_demo.py::TestHashProbingDetection::test_hash_probing_adaptive_pow_increases_difficulty \n[adapt

In [ ]:
run_and_show([REPO_PYTHON, '-m', 'pytest', 'tests/test_encryption.py', '-q'])

COMMAND: E:\secure-dedup\.venv\Scripts\python.exe -m pytest tests/test_encryption.py -q
RETURN CODE: 0
.....                                                                    [100%]
5 passed in 0.19s



CompletedProcess(args=['E:\\secure-dedup\\.venv\\Scripts\\python.exe', '-m', 'pytest', 'tests/test_encryption.py', '-q'], returncode=0, stdout='\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b                                                                    [100%]\x1b\n\x1b\x1b\x1b5 passed\x1b\x1b in 0.19s\x1b\x1b\n', stderr='')

## Benchmark Script Transparency

This is the actual script that generated the encryption comparison artifact.

In [23]:
run_and_show([REPO_PYTHON, 'compare_dedup_encryption_schemes.py', '--print-table'])

COMMAND: E:\secure-dedup\.venv\Scripts\python.exe compare_dedup_encryption_schemes.py --print-table
RETURN CODE: 0

ENCRYPTION SCHEME COMPARISON
Dataset: 300 chunks (90 unique), 5-round average
+-----------------------------+------------------+--------------------+
| Metric                      | Baseline         | Proposed           |
+-----------------------------+------------------+--------------------+
| Dedup saved (%)             | 70.00%           | 70.00% (IDENTICAL) |
| Token generation (ms)       | 0.008519         | 0.013566 (+59.2%)  |
| Encryption time (ms)        | 0.076468         | 0.079984 (+4.6%)   |
| Decryption time (ms)        | 0.084367         | 0.089534 (+6.1%)   |
| Storage overhead delta      | 0 bytes          | +0.0 bytes         |
| Token reproducible?         | YES (vulnerable) | NO (HMAC required) |  <-- KEY
| Frequency attack resistant? | NO               | YES                |  <-- KEY
| External key server?        | NO               | NO (vs REFA: YES)

CompletedProcess(args=['E:\\secure-dedup\\.venv\\Scripts\\python.exe', 'compare_dedup_encryption_schemes.py', '--print-table'], returncode=0, stdout='\nENCRYPTION SCHEME COMPARISON\nDataset: 300 chunks (90 unique), 5-round average\n+-----------------------------+------------------+--------------------+\n| Metric                      | Baseline         | Proposed           |\n+-----------------------------+------------------+--------------------+\n| Dedup saved (%)             | 70.00%           | 70.00% (IDENTICAL) |\n| Token generation (ms)       | 0.008519         | 0.013566 (+59.2%)  |\n| Encryption time (ms)        | 0.076468         | 0.079984 (+4.6%)   |\n| Decryption time (ms)        | 0.084367         | 0.089534 (+6.1%)   |\n| Storage overhead delta      | 0 bytes          | +0.0 bytes         |\n| Token reproducible?         | YES (vulnerable) | NO (HMAC required) |  <-- KEY\n| Frequency attack resistant? | NO               | YES                |  <-- KEY\n| External key serve

In [24]:
comparison = json.loads((REPO_ROOT / 'docs' / 'project_notes' / 'encryption_scheme_comparison.json').read_text(encoding='utf-8'))

comparison_rows = []
for scheme in comparison['schemes']:
    comparison_rows.append({
        'scheme': scheme['scheme'],
        'dedup_saved_percent': scheme['dedup_saved_percent'],
        'avg_token_time_ms': scheme['avg_token_time_ms'],
        'avg_encrypt_time_ms': scheme['avg_encrypt_time_ms'],
        'avg_decrypt_time_ms': scheme['avg_decrypt_time_ms'],
        'token_reproducible_without_secret': scheme['security_properties']['token_reproducible_without_secret'],
        'frequency_attack_resistant': scheme['security_properties']['frequency_attack_resistant'],
        'external_key_server_required': scheme['security_properties']['external_key_server_required'],
    })

display(pd.DataFrame(comparison_rows))
print('Delta summary:')
print(json.dumps(comparison['comparison'], indent=2))

,scheme,dedup_saved_percent,avg_token_time_ms,avg_encrypt_time_ms,avg_decrypt_time_ms,token_reproducible_without_secret,frequency_attack_resistant,external_key_server_required
0,baseline_sha256_bound_aead,70.0,0.008519,0.076468,0.084367,True,False,False
1,proposed_secret_hmac_bound_aead,70.0,0.013566,0.079984,0.089534,False,True,False


Delta summary:
{
  "dedup_saved_delta_pct": 0.0,
  "token_time_delta_pct": 59.244043,
  "encrypt_time_delta_pct": 4.598002,
  "decrypt_time_delta_pct": 6.124433,
  "storage_overhead_delta_bytes": 0.0
}


## Live API Demo: Controlled Similar Files, PoW, and Shared Chunks

The file construction below deliberately keeps two large common regions with different middle regions, so shared chunks are visible and repeatable.

In [39]:
CHUNK_SIZE = 16384

def build_block(tag: str, size: int = CHUNK_SIZE) -> bytes:
    raw = (tag * ((size // len(tag)) + 2)).encode('ascii')
    return raw[:size]

def upload_with_optional_pow(path: Path, pow_proofs_json=None):
    data = {}
    if pow_proofs_json is not None:
        data['pow_proofs_json'] = json.dumps(pow_proofs_json)
    with path.open('rb') as fh:
        response = requests.post(
            f'{BASE_URL}/upload',
            headers=HEADERS,
            files={'file': (path.name, fh, 'text/plain')},
            data=data,
            timeout=180,
        )
    return response

def solve_pow(challenges):
    payload = {
        'challenges': [
            {
                'chunk_hash': item['chunk_hash'],
                'challenge_id': item['challenge_id'],
                'nonce_hex': item['nonce_hex'],
                'offset': item['offset'],
                'length': item['length'],
            }
            for item in challenges
        ]
    }
    response = requests.post(
        f'{BASE_URL}/demo/solve_pow',
        headers={'X-API-Key': API_KEY},
        json=payload,
        timeout=180,
    )
    response.raise_for_status()
    return response.json()['pow_proofs']

session = str(int(time.time()))
common_left = build_block(f'COMMON-LEFT-{session}-')
variant_a = build_block(f'VARIANT-A-{session}-')
variant_b = build_block(f'VARIANT-B-{session}-')
common_right = build_block(f'COMMON-RIGHT-{session}-')

file_a_bytes = common_left + variant_a + common_right
file_b_bytes = common_left + variant_b + common_right

work = Path(tempfile.mkdtemp(prefix='secure-dedup-notebook-'))
path_a = work / 'similar_a.txt'
path_b = work / 'similar_b.txt'
path_a.write_bytes(file_a_bytes)
path_b.write_bytes(file_b_bytes)

upload_a = upload_with_optional_pow(path_a)
upload_a.raise_for_status()
body_a = upload_a.json()

upload_b_first = upload_with_optional_pow(path_b)
body_b_first = upload_b_first.json()

if upload_b_first.status_code == 409:
    proofs = solve_pow(body_b_first['detail']['required_challenges'])
    upload_b = upload_with_optional_pow(path_b, pow_proofs_json=proofs)
else:
    proofs = {}
    upload_b = upload_b_first

upload_b.raise_for_status()
body_b = upload_b.json()

compare = requests.get(
    f'{BASE_URL}/demo/compare-files',
    headers=HEADERS,
    params={
        'file_id_a': body_a['file']['file_id'],
        'file_id_b': body_b['file']['file_id'],
    },
    timeout=180,
)
compare.raise_for_status()
compare_body = compare.json()['comparison']

print('Upload A chunk summary:')
print(json.dumps(body_a['chunk_summary'], indent=2))
print('\nUpload B first response status:', upload_b_first.status_code)
print(json.dumps(body_b_first, indent=2))
print('\nUpload B final chunk summary:')
print(json.dumps(body_b['chunk_summary'], indent=2))
print('\nComputed PoW proofs:')
print(json.dumps(proofs, indent=2))

shared_positions_df = pd.DataFrame(compare_body['shared_chunk_positions'])
display(shared_positions_df)
print('\nShared chunk count =', compare_body['shared_chunk_count'])
print('Interpretation =', compare_body['interpretation'])

Upload A chunk summary:
{
  "logical_chunk_count": 3,
  "unique_chunk_count": 3,
  "shared_with_existing_count": 0,
  "new_chunk_count": 3,
  "reused_existing_count": 0,
  "pow_required_count": 0,
  "pow_verified_count": 0,
  "shared_chunk_hashes": [],
  "unique_chunk_hashes": [
    "fd40acaa56008385208e28b623c8c5be0b85d454cd029e14d7d97f0b40606751",
    "f4984fd5f519fdd44de2e7b0e3a1c74f97fc5d3483b0f4a18e673d8d930cc362",
    "7aa6a1ad993ece729ab0b14caf1644536b038e27a53d85c9e7df61a84daf2780"
  ]
}

Upload B first response status: 409
{
  "detail": {
    "error": "PoW verification required for duplicate chunks",
    "client_id": "notebook-demo-1774323266",
    "required_challenges": [
      {
        "chunk_hash": "fd40acaa56008385208e28b623c8c5be0b85d454cd029e14d7d97f0b40606751",
        "challenge_id": "afaa9283-d1fa-4ac4-8135-ae102cad7f8c",
        "nonce_hex": "8cfccb93399ec4be48750dd3261b9282",
        "offset": 9794,
        "length": 61,
        "expires_at": 1774332107.2196672,
  

,chunk_hash,positions_in_file_a,positions_in_file_b
0,7aa6a1ad993ece729ab0b14caf1644536b038e27a53d85...,[2],[2]
1,fd40acaa56008385208e28b623c8c5be0b85d454cd029e...,[0],[0]



Shared chunk count = 2
Interpretation = These files share chunk hashes and can visibly demonstrate dedup reuse.


## Live API Demo: Forced Rate Limit and Highlights

In [38]:
force = requests.post(
    f'{BASE_URL}/demo/force-policy',
    headers={'X-API-Key': API_KEY},
    json={'client_id': CLIENT_ID, 'action': 'RATE_LIMIT'},
    timeout=60,
)
force.raise_for_status()

attack_path = work / 'attack_check.txt'
attack_path.write_text('attack-check-' + session, encoding='utf-8')
with attack_path.open('rb') as fh:
    blocked = requests.post(
        f'{BASE_URL}/upload',
        headers=HEADERS,
        files={'file': (attack_path.name, fh, 'text/plain')},
        timeout=180,
    )

blocked_body = blocked.json()
highlights = requests.get(
    f'{BASE_URL}/demo/highlights/{CLIENT_ID}',
    headers={'X-API-Key': API_KEY},
    timeout=180,
)
highlights.raise_for_status()
highlights_body = highlights.json()

clear = requests.post(
    f'{BASE_URL}/demo/clear-policy',
    headers={'X-API-Key': API_KEY},
    json={'client_id': CLIENT_ID},
    timeout=60,
)
clear.raise_for_status()

print('Forced policy response:')
print(json.dumps(force.json(), indent=2))
print('\nBlocked upload status:', blocked.status_code)
print(json.dumps(blocked_body, indent=2))
print('\nHighlights summary:')
print(json.dumps(highlights_body['highlights'], indent=2))
display(pd.DataFrame(highlights_body['recent_events']))

Forced policy response:
{
  "client_id": "notebook-demo-1774323266",
  "action": "RATE_LIMIT",
  "active_policy": {
    "action": "RATE_LIMIT",
    "expires_at": 1774332005.3685143,
    "ttl_sec": 30,
    "remaining_sec": 29.971160411834717
  }
}

Blocked upload status: 429
{
  "detail": {
    "error": "Rate limited by anomaly policy",
    "client_id": "notebook-demo-1774323266",
    "policy": {
      "action": "RATE_LIMIT",
      "status_code": 429,
      "remaining_sec": 29.920063495635986
    }
  }
}

Highlights summary:
{
  "upload_attempts": 3,
  "stored_new_chunks": 4,
  "pow_challenges_issued": 2,
  "pow_verifications": 2,
  "duplicate_reuse_successes": 2,
  "rate_limit_events": 1,
  "block_events": 0
}


,timestamp,operation_type,chunk_hash,pow_result
0,1.774324e+09,pow_challenge,0f5cd9a48b3884a9bc73ab1fe7c018cebeb0c960f2d3f4...,required
1,1.774324e+09,pow_challenge,78d98d7931e68e08ac57661decd14d240973c04fa977da...,required
2,1.774324e+09,upload_start,NaN,None
3,1.774324e+09,pow_verify,0f5cd9a48b3884a9bc73ab1fe7c018cebeb0c960f2d3f4...,True
4,1.774324e+09,pow_verify,78d98d7931e68e08ac57661decd14d240973c04fa977da...,True
5,1.774324e+09,hash_query,0f5cd9a48b3884a9bc73ab1fe7c018cebeb0c960f2d3f4...,None
6,1.774324e+09,pow,0f5cd9a48b3884a9bc73ab1fe7c018cebeb0c960f2d3f4...,True
7,1.774324e+09,hash_query,f3135ec7b1b1f43c22b716d8fef8efea46b88eaf1ac000...,None
8,1.774324e+09,upload_chunk,f3135ec7b1b1f43c22b716d8fef8efea46b88eaf1ac000...,N/A
9,1.774324e+09,hash_query,78d98d7931e68e08ac57661decd14d240973c04fa977da...,None


## Dataset and Model Transparency

In [27]:
training_metrics = json.loads((REPO_ROOT / 'dense_artifacts' / 'training_metrics.json').read_text(encoding='utf-8'))
summary = {
    'rows': training_metrics['rows'],
    'train_rows': training_metrics['train_rows'],
    'test_rows': training_metrics['test_rows'],
    'best_model': training_metrics['best_model'],
    'best_cv_score': training_metrics['best_cv_score'],
}
print(json.dumps(summary, indent=2))

class_distribution = pd.DataFrame(
    list(training_metrics['class_distribution'].items()),
    columns=['attack_label', 'rows'],
).sort_values('rows', ascending=False)
display(class_distribution)

candidate_results = pd.DataFrame(training_metrics['candidate_results']).sort_values('best_cv_score', ascending=False)
display(candidate_results[['model', 'best_cv_score']])

{
  "rows": 221,
  "train_rows": 165,
  "test_rows": 56,
  "best_model": "random_forest",
  "best_cv_score": 0.9652154354806236
}


,attack_label,rows
0,ownership_fraud,120
1,hash_probing,53
2,normal,32
3,dedup_dos,16


,model,best_cv_score
1,random_forest,0.965215
3,extra_trees,0.915287
0,hist_gradient_boosting,0.891976
2,logistic_regression,0.826629
4,svc_rbf,0.799859
5,mlp_classifier,0.666277


In [28]:
display(Markdown((REPO_ROOT / 'dense_artifacts' / 'evaluation_report.md').read_text(encoding='utf-8')))

# Model Evaluation Report

- Generated At (UTC): `2026-03-22T12:36:25.825642+00:00`
- Dataset: `multisource_dense_detection_results.csv`
- Model Dir: `dense_artifacts`
- Training Metadata Mode: `supervised`
- Evaluated Mode: `supervised`
- Rows Evaluated: `221`

## Binary Metrics (normal vs anomaly)
- PR-AUC: `1.000000`
- F1 (binary): `1.000000`
- F1 (macro): `1.000000`
- Precision: `1.000000`
- Recall: `1.000000`

| Truth \ Pred | normal | anomaly |
|---|---:|---:|
| normal | 32 | 0 |
| anomaly | 0 | 189 |

## Multiclass Metrics
- Labels: `dedup_dos, hash_probing, normal, ownership_fraud`
- F1 (macro): `1.000000`
- PR-AUC (OvR macro): `1.0`

Confusion matrix (JSON):
```json
[
  [
    16,
    0,
    0,
    0
  ],
  [
    0,
    53,
    0,
    0
  ],
  [
    0,
    0,
    32,
    0
  ],
  [
    0,
    0,
    0,
    120
  ]
]
```

## Prediction Summary
- Predicted Normal: `32`
- Predicted Anomaly: `189`
- Predicted Anomaly Rate: `0.855204`
- Mean Risk Score: `0.851733`
